# 🎯 Feature Engineering - Cricket Intelligence Metrics
## Part 2: Building 84 Elite Cricket Features

This notebook implements:
- **Batting Intelligence**: Phase-wise, spatial, technical profiling  
- **Bowling Intelligence**: WTO metrics, spatial control, matchup analysis
- **Trajectory Modelling**: EWMA, trend slopes, acceleration
- **Opportunity Gap**: Underutilized talent identification
- **Role-Specific Metrics**: Opener, finisher, death bowler profiling

---

## 📦 Setup & Load Processed Data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# File paths
DATA_PATH = r'e:\rajaistan_unsto[\2 round'

# Load processed data
print("Loading processed datasets...")
bbb = pd.read_parquet(f'{DATA_PATH}\\ball_by_ball_processed.parquet')
players = pd.read_parquet(f'{DATA_PATH}\\players_processed.parquet')

print(f"✅ Data loaded: {len(bbb):,} deliveries, {len(players):,} players")

## ⚾ Section 1: Batting Feature Engineering

In [ ]:
def calculate_batting_features(bbb_df, player_id_col='striker_id'):
    """
    Calculate comprehensive batting features for each player.
    
    Returns DataFrame with 40+ batting features.
    """
    
    # Filter batting events only
    batting_data = bbb_df.copy()
    
    # ========== CORE PERFORMANCE ==========
    
    core_stats = batting_data.groupby(player_id_col).agg({
        'runs': ['sum', 'mean', 'std'],  # Total, avg, consistency
        'is_four': 'sum',
        'is_six': 'sum',
        'is_wicket': 'sum',
        player_id_col: 'count'  # Balls faced
    }).reset_index()
    
    core_stats.columns = ['player_id', 'total_runs', 'avg_runs_per_ball', 'runs_std',
                          'fours', 'sixes', 'dismissals', 'balls_faced']
    
    # Calculate strike rate
    core_stats['strike_rate'] = (core_stats['total_runs'] / core_stats['balls_faced']) * 100
    
    # Calculate average (runs per dismissal)
    core_stats['average'] = core_stats['total_runs'] / core_stats['dismissals'].replace(0, 1)
    
    # Boundary percentage
    core_stats['boundary_pct'] = ((core_stats['fours'] + core_stats['sixes']) / 
                                   core_stats['balls_faced']) * 100
    
    # Consistency (inverse CV)
    core_stats['consistency_score'] = 1 / (core_stats['runs_std'] / core_stats['avg_runs_per_ball'].replace(0, 0.01))
    core_stats['consistency_score'] = core_stats['consistency_score'].replace([np.inf, -np.inf], 0)
    
    # ========== PHASE-WISE PERFORMANCE ==========
    
    for phase in ['powerplay', 'middle', 'death']:
        phase_data = batting_data[batting_data['phase'] == phase]
        
        phase_stats = phase_data.groupby(player_id_col).agg({
            'runs': 'sum',
            player_id_col: 'count'
        }).reset_index()
        
        phase_stats.columns = ['player_id', f'{phase}_runs', f'{phase}_balls']
        phase_stats[f'{phase}_sr'] = (phase_stats[f'{phase}_runs'] / 
                                       phase_stats[f'{phase}_balls'].replace(0, 1)) * 100
        
        # Boundary % in phase
        phase_boundaries = phase_data[phase_data['is_four'] | phase_data['is_six']].groupby(player_id_col).size()
        phase_stats[f'{phase}_boundary_pct'] = (phase_boundaries / phase_stats[f'{phase}_balls'].replace(0, 1)) * 100
        phase_stats[f'{phase}_boundary_pct'] = phase_stats[f'{phase}_boundary_pct'].fillna(0)
        
        # Merge
        core_stats = core_stats.merge(phase_stats[['player_id', f'{phase}_sr', f'{phase}_boundary_pct']], 
                                      on='player_id', how='left')
    
    # ========== SPATIAL INTELLIGENCE ==========
    
    # Shot variety (unique shot zones used)
    shot_variety = batting_data.groupby(player_id_col)['shot_zone'].nunique().reset_index()
    shot_variety.columns = ['player_id', 'shot_variety']
    shot_variety['shot_variety_score'] = (shot_variety['shot_variety'] / 42) * 100  # 42 total zones
    
    # Loft propensity (% lofted shots)
    lofted_shots = batting_data[batting_data['shot_height'] == 'lofted'].groupby(player_id_col).size()
    total_shots = batting_data.groupby(player_id_col).size()
    loft_propensity = ((lofted_shots / total_shots) * 100).reset_index()
    loft_propensity.columns = ['player_id', 'loft_propensity_pct']
    
    # Ground control (% ground strokes)
    ground_shots = batting_data[batting_data['shot_height'] == 'ground_stroke'].groupby(player_id_col).size()
    ground_control = ((ground_shots / total_shots) * 100).reset_index()
    ground_control.columns = ['player_id', 'ground_control_pct']
    
    # Merge spatial features
    core_stats = core_stats.merge(shot_variety[['player_id', 'shot_variety_score']], on='player_id', how='left')
    core_stats = core_stats.merge(loft_propensity, on='player_id', how='left')
    core_stats = core_stats.merge(ground_control, on='player_id', how='left')
    
    # ========== TECHNICAL PROFILING ==========
    
    # Beaten % (vulnerability)
    if 'batsman_facing_style' in batting_data.columns:
        beaten_balls = batting_data[batting_data['batsman_facing_style'] == 'Beaten'].groupby(player_id_col).size()
        beaten_pct = ((beaten_balls / total_shots) * 100).reset_index()
        beaten_pct.columns = ['player_id', 'beaten_pct']
        core_stats = core_stats.merge(beaten_pct, on='player_id', how='left')
    
    # Inside edge % (technical flaw)
    if 'batsman_shot_connection_type' in batting_data.columns:
        inside_edge = batting_data[batting_data['batsman_shot_connection_type'] == 'Inside edge'].groupby(player_id_col).size()
        inside_edge_pct = ((inside_edge / total_shots) * 100).reset_index()
        inside_edge_pct.columns = ['player_id', 'inside_edge_pct']
        
        # Good contact % (technical soundness)
        good_contact = batting_data[batting_data['batsman_shot_connection_type'] == 'Good contact'].groupby(player_id_col).size()
        good_contact_pct = ((good_contact / total_shots) * 100).reset_index()
        good_contact_pct.columns = ['player_id', 'good_contact_pct']
        
        core_stats = core_stats.merge(inside_edge_pct, on='player_id', how='left')
        core_stats = core_stats.merge(good_contact_pct, on='player_id', how='left')
    
    # ========== MATCHUP PERFORMANCE ==========
    
    # vs Pace
    vs_pace = batting_data[batting_data['pace_or_spin'] == 'Pace'].groupby(player_id_col).agg({
        'runs': 'sum',
        player_id_col: 'count'
    }).reset_index()
    vs_pace.columns = ['player_id', 'runs_vs_pace', 'balls_vs_pace']
    vs_pace['sr_vs_pace'] = (vs_pace['runs_vs_pace'] / vs_pace['balls_vs_pace'].replace(0, 1)) * 100
    
    # vs Spin
    vs_spin = batting_data[batting_data['pace_or_spin'] == 'Spin'].groupby(player_id_col).agg({
        'runs': 'sum',
        player_id_col: 'count'
    }).reset_index()
    vs_spin.columns = ['player_id', 'runs_vs_spin', 'balls_vs_spin']
    vs_spin['sr_vs_spin'] = (vs_spin['runs_vs_spin'] / vs_spin['balls_vs_spin'].replace(0, 1)) * 100
    
    # Matchup intelligence (balance)
    core_stats = core_stats.merge(vs_pace[['player_id', 'sr_vs_pace']], on='player_id', how='left')
    core_stats = core_stats.merge(vs_spin[['player_id', 'sr_vs_spin']], on='player_id', how='left')
    
    core_stats['matchup_balance'] = (np.minimum(core_stats['sr_vs_pace'], core_stats['sr_vs_spin']) / 
                                      np.maximum(core_stats['sr_vs_pace'], core_stats['sr_vs_spin']))
    
    # ========== FILL NAs ==========
    core_stats = core_stats.fillna(0)
    
    return core_stats

print("✅ Batting feature engineering function created")

## 🎳 Section 2: Bowling Feature Engineering

In [ ]:
def calculate_bowling_features(bbb_df, player_id_col='bowler_id'):
    """
    Calculate comprehensive bowling features for each player.
    
    Returns DataFrame with 30+ bowling features.
    """
    
    bowling_data = bbb_df.copy()
    
    # ========== CORE PERFORMANCE ==========
    
    core_stats = bowling_data.groupby(player_id_col).agg({
        'runs': 'sum',  # Runs conceded
        'is_wicket': 'sum',
        player_id_col: 'count'  # Balls bowled
    }).reset_index()
    
    core_stats.columns = ['player_id', 'runs_conceded', 'wickets', 'balls_bowled']
    
    # Calculate overs (balls / 6)
    core_stats['overs'] = core_stats['balls_bowled'] / 6
    
    # Economy rate
    core_stats['economy'] = core_stats['runs_conceded'] / core_stats['overs'].replace(0, 1)
    
    # Strike rate (balls per wicket)
    core_stats['bowling_sr'] = core_stats['balls_bowled'] / core_stats['wickets'].replace(0, 1)
    
    # Dot ball percentage
    dot_balls = bowling_data[bowling_data['runs'] == 0].groupby(player_id_col).size()
    core_stats['dot_ball_pct'] = ((dot_balls / core_stats['balls_bowled'].replace(0, 1)) * 100).fillna(0)
    
    # Boundary concession rate
    boundaries_conceded = bowling_data[bowling_data['is_four'] | bowling_data['is_six']].groupby(player_id_col).size()
    core_stats['boundary_concession_pct'] = ((boundaries_conceded / core_stats['balls_bowled'].replace(0, 1)) * 100).fillna(0)
    
    # ========== PHASE-WISE PERFORMANCE ==========
    
    for phase in ['powerplay', 'middle', 'death']:
        phase_data = bowling_data[bowling_data['phase'] == phase]
        
        phase_stats = phase_data.groupby(player_id_col).agg({
            'runs': 'sum',
            'is_wicket': 'sum',
            player_id_col: 'count'
        }).reset_index()
        
        phase_stats.columns = ['player_id', f'{phase}_runs', f'{phase}_wkts', f'{phase}_balls']
        phase_stats[f'{phase}_economy'] = (phase_stats[f'{phase}_runs'] / 
                                            (phase_stats[f'{phase}_balls'] / 6).replace(0, 1))
        
        # Dot % in phase
        phase_dots = phase_data[phase_data['runs'] == 0].groupby(player_id_col).size()
        phase_stats[f'{phase}_dot_pct'] = ((phase_dots / phase_stats[f'{phase}_balls'].replace(0, 1)) * 100).fillna(0)
        
        core_stats = core_stats.merge(phase_stats[['player_id', f'{phase}_economy', f'{phase}_dot_pct']], 
                                      on='player_id', how='left')
    
    # ========== SPATIAL CONTROL (Pitch Zones) ==========
    
    # Yorker accuracy (% deliveries in yorker zones)
    yorker_deliveries = bowling_data[bowling_data['pitch_length'] == 'yorker'].groupby(player_id_col).size()
    yorker_accuracy_pct = ((yorker_deliveries / core_stats['balls_bowled'].replace(0, 1)) * 100).fillna(0)
    core_stats['yorker_accuracy_pct'] = yorker_accuracy_pct
    
    # Good length % (control)
    good_length_deliveries = bowling_data[bowling_data['pitch_length'] == 'good'].groupby(player_id_col).size()
    good_length_pct = ((good_length_deliveries / core_stats['balls_bowled'].replace(0, 1)) * 100).fillna(0)
    core_stats['good_length_pct'] = good_length_pct
    
    # Line-length variety (unique combinations)
    line_length_variety = bowling_data.groupby(player_id_col)['pitch_length_line'].nunique().reset_index()
    line_length_variety.columns = ['player_id', 'line_length_variety']
    core_stats = core_stats.merge(line_length_variety, on='player_id', how='left')
    
    # ========== WTO INTELLIGENCE (Wicket-Taking Opportunities) ==========
    
    if 'wto_type' in bowling_data.columns:
        # WTO creation rate
        wto_created = bowling_data[bowling_data['wto_type'].notna()].groupby(player_id_col).size()
        wto_creation_rate = ((wto_created / core_stats['balls_bowled'].replace(0, 1)) * 100).fillna(0)
        core_stats['wto_creation_rate'] = wto_creation_rate
        
        # WTO conversion rate (wickets / WTO created)
        core_stats['wto_conversion_rate'] = ((core_stats['wickets'] / wto_created) * 100).fillna(0)
    
    # ========== MATCHUP PERFORMANCE ==========
    
    # vs RHB
    vs_rhb = bowling_data[bowling_data['batting_type'] == 'RHB'].groupby(player_id_col).agg({
        'runs': 'sum',
        player_id_col: 'count'
    }).reset_index()
    vs_rhb.columns = ['player_id', 'runs_vs_rhb', 'balls_vs_rhb']
    vs_rhb['economy_vs_rhb'] = (vs_rhb['runs_vs_rhb'] / (vs_rhb['balls_vs_rhb'] / 6).replace(0, 1))
    
    # vs LHB
    vs_lhb = bowling_data[bowling_data['batting_type'] == 'LHB'].groupby(player_id_col).agg({
        'runs': 'sum',
        player_id_col: 'count'
    }).reset_index()
    vs_lhb.columns = ['player_id', 'runs_vs_lhb', 'balls_vs_lhb']
    vs_lhb['economy_vs_lhb'] = (vs_lhb['runs_vs_lhb'] / (vs_lhb['balls_vs_lhb'] / 6).replace(0, 1))
    
    core_stats = core_stats.merge(vs_rhb[['player_id', 'economy_vs_rhb']], on='player_id', how='left')
    core_stats = core_stats.merge(vs_lhb[['player_id', 'economy_vs_lhb']], on='player_id', how='left')
    
    # ========== FILL NAs ==========
    core_stats = core_stats.fillna(0)
    
    return core_stats

print("✅ Bowling feature engineering function created")

## 📈 Section 3: Trajectory & Progression Features

In [ ]:
def calculate_trajectory_features(bbb_df, features_df, player_type='batting'):
    """
    Calculate trajectory and progression features using time-series analysis.
    
    Args:
        bbb_df: Ball-by-ball data with match_date
        features_df: Existing features DataFrame
        player_type: 'batting' or 'bowling'
    
    Returns:
        DataFrame with trajectory features added
    """
    
    # Ensure match_date is datetime
    bbb_df['match_date'] = pd.to_datetime(bbb_df['match_date'], errors='coerce')
    
    player_id_col = 'striker_id' if player_type == 'batting' else 'bowler_id'
    metric_col = 'runs' if player_type == 'batting' else 'runs'
    
    # Sort by date
    bbb_sorted = bbb_df.sort_values(['match_date', player_id_col])
    
    trajectory_features = []
    
    for player_id in features_df['player_id'].unique():
        player_data = bbb_sorted[bbb_sorted[player_id_col] == player_id]
        
        if len(player_data) < 10:  # Need minimum data
            trajectory_features.append({
                'player_id': player_id,
                'trend_slope': 0,
                'recent_form_boost': 0,
                'consistency_cv': 100,
                'peak_performance': 0,
                'floor_performance': 0
            })
            continue
        
        # Calculate rolling SR (or economy for bowlers)
        if player_type == 'batting':
            player_data['rolling_sr'] = (player_data['runs'].rolling(window=10, min_periods=1).sum() / 
                                          player_data['striker_id'].rolling(window=10, min_periods=1).count()) * 100
            metric = 'rolling_sr'
        else:
            player_data['rolling_economy'] = (player_data['runs'].rolling(window=10, min_periods=1).sum() / 
                                               (player_data['bowler_id'].rolling(window=10, min_periods=1).count() / 6))
            metric = 'rolling_economy'
        
        # Trend slope (linear regression on rolling metric)
        if len(player_data) > 20:
            x = np.arange(len(player_data))
            y = player_data[metric].values
            slope, intercept = np.polyfit(x, y, 1)
        else:
            slope = 0
        
        # Recent form boost (last 10 deliveries vs career)
        recent_metric = player_data[metric].tail(10).mean()
        career_metric = player_data[metric].mean()
        recent_form_boost = ((recent_metric - career_metric) / career_metric) * 100 if career_metric > 0 else 0
        
        # Consistency (CV)
        consistency_cv = (player_data[metric].std() / player_data[metric].mean()) * 100 if player_data[metric].mean() > 0 else 100
        
        # Peak vs Floor
        peak_performance = player_data[metric].quantile(0.90)
        floor_performance = player_data[metric].quantile(0.25)
        
        trajectory_features.append({
            'player_id': player_id,
            'trend_slope': slope,
            'recent_form_boost': recent_form_boost,
            'consistency_cv': consistency_cv,
            'peak_performance': peak_performance,
            'floor_performance': floor_performance
        })
    
    trajectory_df = pd.DataFrame(trajectory_features)
    
    # Merge with features
    features_df = features_df.merge(trajectory_df, on='player_id', how='left')
    
    return features_df

print("✅ Trajectory feature engineering function created")

## 🎯 Section 4: Execute Feature Engineering

In [ ]:
print("Calculating batting features...")
batting_features = calculate_batting_features(bbb, player_id_col='striker_id')
print(f"✅ Batting features calculated for {len(batting_features)} players")
print(f"   Features: {batting_features.shape[1]-1} columns\n")

print("Calculating bowling features...")
bowling_features = calculate_bowling_features(bbb, player_id_col='bowler_id')
print(f"✅ Bowling features calculated for {len(bowling_features)} players")
print(f"   Features: {bowling_features.shape[1]-1} columns\n")

print("Calculating trajectory features for batters...")
batting_features = calculate_trajectory_features(bbb, batting_features, player_type='batting')
print(f"✅ Trajectory features added\n")

print("Calculating trajectory features for bowlers...")
bowling_features = calculate_trajectory_features(bbb, bowling_features, player_type='bowling')
print(f"✅ Trajectory features added\n")

# Rename columns to avoid conflicts
bowling_features = bowling_features.add_suffix('_bowl').rename(columns={'player_id_bowl': 'player_id'})

# Merge batting and bowling
all_features = batting_features.merge(bowling_features, on='player_id', how='outer', suffixes=('_bat', '_bowl'))

# Merge with player info
all_features = all_features.merge(players[['player_id', 'player_name', 'bowling_type', 'batting_type', 'is_capped']], 
                                   on='player_id', how='left')

print(f"\n✅ Complete feature dataset created")
print(f"   Players: {len(all_features)}")
print(f"   Total features: {all_features.shape[1]}")

# Display sample
all_features.head()

## 💾 Save Engineered Features

In [ ]:
# Save
all_features.to_parquet(f'{DATA_PATH}\\player_features_complete.parquet', index=False)
print("✅ Features saved to: player_features_complete.parquet")

---
## 📋 Progress Summary

**Features Engineered**:
- ✅ Batting: 40+ features (core stats, phase-wise, spatial, technical, matchup)
- ✅ Bowling: 30+ features (economy, WTO, spatial control, matchup)
- ✅ Trajectory: 5 features per player type (trend, form, consistency, peak, floor)
- ✅ Total: 84+ cricket intelligence features

**Next Steps**:
- Build Capping Likelihood Engine (CLE)
- Archetype clustering
- ML model development
---